# Whisper Speech Recognition Integration

Pipeline: record or load audio → transcribe with Whisper (local GPU) → pass transcript to agent → get answer.
This gives the bot true multimodal capability — users can ask questions by voice instead of typing.

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path("../.env"))

AUDIO_DIR = Path("../data/audio")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete.")
print(f"Audio directory ready: {AUDIO_DIR.resolve().name}")

Setup complete.
Audio directory ready: audio


## 1. Load Whisper Model

We load the `medium` model locally — good balance of speed and accuracy for English and Spanish.
Running on GPU via CUDA means transcription takes only a few seconds per clip.

In [2]:
import torch
import whisper

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = whisper.load_model("medium", device=device)
print("Whisper model loaded.")

Using device: cuda
Whisper model loaded.


## 2. Transcribe an Audio File

Pass any audio file (mp3, wav, m4a) to Whisper. It returns the full transcription along with
detected language — useful for handling multilingual content.

In [3]:
def transcribe_audio(file_path: str) -> dict:
    audio_path = Path(file_path)
    
    if not audio_path.exists():
        print(f"File not found: {file_path}")
        return {}
    
    print(f"Transcribing: {audio_path.name}")
    result = model.transcribe(str(audio_path), fp16=torch.cuda.is_available())
    
    print(f"Detected language: {result['language']}")
    print(f"Transcription length: {len(result['text'])} chars")
    
    return {
        "text": result["text"],
        "language": result["language"],
        "segments": result["segments"],
    }

## 3. Download a Test Audio Clip

We use yt-dlp to download audio directly from YouTube for testing.
This simulates a real user submitting a video URL for the bot to process.

In [4]:
import subprocess
import sys

def download_audio(youtube_url: str, output_name: str) -> str:
    output_path = AUDIO_DIR / f"{output_name}.mp3"
    
    if output_path.exists():
        print(f"Already downloaded: {output_name}.mp3")
        return str(output_path)
    
    print(f"Downloading audio: {output_name}")
    
    yt_dlp_path = Path(sys.executable).parent / "yt-dlp"
    
    subprocess.run([
        str(yt_dlp_path),
        "-x",
        "--audio-format", "mp3",
        "--audio-quality", "0",
        "-o", str(output_path),
        youtube_url,
    ], check=True)
    
    print(f"Saved to: {output_path.name}")
    return str(output_path)

audio_path = download_audio(
    "https://www.youtube.com/watch?v=u-ZBLZATlhM",
    "erwin_speech_test"
)

Already downloaded: erwin_speech_test.mp3


## 4. Run Transcription

Transcribe the downloaded clip and compare Whisper's output against the 
auto-generated YouTube captions we used in Day 1.

In [5]:
result = transcribe_audio(audio_path)

print("\nWhisper Transcription:")
print("-" * 60)
print(result["text"])

Transcribing: erwin_speech_test.mp3
Detected language: en
Transcription length: 1298 chars

Whisper Transcription:
------------------------------------------------------------
 There's no point standing around. We'll only be showered by more boulders. Ready your horses on the double! Be honest. Are all of us... riding to our deaths? Yes, we are. And since we're dying anyway, you're saying that it's better... if we at least stop fighting? I am. But wait... if we'll die anyway... then who cares what we do? We could just disobey your orders... and it wouldn't mean a thing, would it? Yes, you're precisely right. Everything that you thought had meaning... every hope, dream, or moment of happiness... none of it matters as you lie bleeding out on the battlefield. None of it changes what a speeding rock does to a body. We all die. But does that mean our lives are meaningless? Does that mean that there was no point in our being born? Would you say that of our slain comrades? What about their li

## 5. Connect Whisper to the Agent

Full pipeline: audio file → Whisper transcription → agent query → answer.
This is the core multimodal flow the web app will use.

In [6]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.tools.retriever import create_retriever_tool
from langchain_core.tools import tool
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langgraph.prebuilt import create_react_agent

# Load vectorstore
VECTORSTORE_DIR = Path("../data/vectorstore")

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.getenv("OPENAI_API_KEY"),
)

vectorstore = Chroma(
    persist_directory=str(VECTORSTORE_DIR),
    embedding_function=embedding_model,
    collection_name="youtube_qa",
)

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=os.getenv("OPENAI_API_KEY"),
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

retriever_tool = create_retriever_tool(
    retriever=retriever,
    name="youtube_transcript_search",
    description=(
        "Search across YouTube video transcripts covering education, tech/AI, and entertainment. "
        "Use this tool whenever the user asks a question about any of the video content."
    ),
)

@tool
def general_knowledge(query: str) -> str:
    """
    Answer general knowledge questions not covered by the YouTube transcripts.
    Use this when the user asks something outside the scope of the video content.
    """
    response = llm.invoke(query)
    return response.content

message_history = ChatMessageHistory()

def get_session_history(session_id: str):
    return message_history

system_prompt = """You are a helpful assistant with access to YouTube video transcripts
covering education, tech/AI, and entertainment topics.

When answering questions:
- Always search the transcripts first for relevant information
- Reference the video title and category when citing information from transcripts
- If the question is outside the scope of the transcripts, use your general knowledge
- Keep answers clear and concise
"""

tools = [retriever_tool, general_knowledge]

agent_executor = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt,
)

agent_with_memory = RunnableWithMessageHistory(
    agent_executor,
    get_session_history,
    input_messages_key="messages",
    history_messages_key="chat_history",
)

print("Agent loaded and ready.")

Agent loaded and ready.


## 6. Voice Question → Agent Answer

Simulate a user asking a question by voice. Whisper transcribes the audio,
the transcription is passed to the agent, and the agent returns an answer.

In [7]:
from langchain_core.messages import HumanMessage

def ask_by_voice(audio_file_path: str, session_id: str = "voice_session") -> str:
    # Step 1 — transcribe audio
    transcription = transcribe_audio(audio_file_path)
    
    if not transcription:
        return "Could not transcribe audio."
    
    question = transcription["text"].strip()
    print(f"Transcribed question: {question}")
    print("-" * 60)
    
    # Step 2 — pass to agent
    response = agent_with_memory.invoke(
        {"messages": [HumanMessage(content=question)]},
        config={"configurable": {"session_id": session_id}},
    )
    
    return response["messages"][-1].content

# Test — use the Erwin audio as the voice input
answer = ask_by_voice(audio_path)
print(f"Agent answer:\n{answer}")

Transcribing: erwin_speech_test.mp3
Detected language: en
Transcription length: 1298 chars
Transcribed question: There's no point standing around. We'll only be showered by more boulders. Ready your horses on the double! Be honest. Are all of us... riding to our deaths? Yes, we are. And since we're dying anyway, you're saying that it's better... if we at least stop fighting? I am. But wait... if we'll die anyway... then who cares what we do? We could just disobey your orders... and it wouldn't mean a thing, would it? Yes, you're precisely right. Everything that you thought had meaning... every hope, dream, or moment of happiness... none of it matters as you lie bleeding out on the battlefield. None of it changes what a speeding rock does to a body. We all die. But does that mean our lives are meaningless? Does that mean that there was no point in our being born? Would you say that of our slain comrades? What about their lives? Were they meaningless? They were not! Their memory serves a

## Re-ingest Erwin and Bad Bunny with Whisper

In [8]:
# Run after all previous cells in 4_whisper.ipynb
# Replaces caption-based chunks for Erwin and Bad Bunny with Whisper transcriptions
import subprocess
import sys
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from pathlib import Path

TRANSCRIPTS_DIR = Path("../data/transcripts")
yt_dlp_path = Path(sys.executable).parent / "yt-dlp"

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)

videos = [
    {
        "video_id": "u-ZBLZATlhM",
        "title":    "erwin_speech_aot",
        "category": "entertainment",
        "url":      "https://www.youtube.com/watch?v=u-ZBLZATlhM",
    },
    {
        "video_id": "HyqEtb5HE50",
        "title":    "bad_bunny_hot_ones",
        "category": "entertainment",
        "url":      "https://www.youtube.com/watch?v=HyqEtb5HE50",
    },
]

for video in videos:
    print(f"\n{video['title']}")

    # Remove old caption chunks
    existing = vectorstore._collection.get(where={"title": video["title"]})
    if existing["ids"]:
        vectorstore._collection.delete(ids=existing["ids"])
        print(f"  Removed {len(existing['ids'])} old chunks")

    # Download audio if needed
    audio_path = AUDIO_DIR / f"{video['title']}.mp3"
    if not audio_path.exists():
        print("  Downloading audio...")
        subprocess.run([
            str(yt_dlp_path),
            "-x", "--audio-format", "mp3", "--audio-quality", "0",
            "-o", str(audio_path), video["url"],
        ], check=True)
    else:
        print("  Audio already on disk")

    # Transcribe
    print("  Transcribing...")
    result = model.transcribe(str(audio_path), fp16=torch.cuda.is_available())
    text = result["text"].strip()
    print(f"  {len(text)} chars — language: {result['language']}")

    # Embed into ChromaDB
    chunks = splitter.split_text(text)
    docs = [
        Document(
            page_content=chunk,
            metadata={
                "title":    video["title"],
                "category": video["category"],
                "video_id": video["video_id"],
                "source":   video["url"],
                "chunk_id": i,
            },
        )
        for i, chunk in enumerate(chunks)
    ]
    vectorstore.add_documents(docs)
    print(f"  Added {len(docs)} chunks")

    # Save transcript to disk
    save_dir = TRANSCRIPTS_DIR / video["category"]
    save_dir.mkdir(parents=True, exist_ok=True)
    with open(save_dir / f"{video['title']}.json", "w") as f:
        json.dump({
            "video_id":   video["video_id"],
            "title":      video["title"],
            "category":   video["category"],
            "transcript": text,
        }, f, indent=2)
    print("  Saved to disk")

print("\nDone. Restart Streamlit to use updated embeddings.")


erwin_speech_aot
  Removed 3 old chunks
[youtube] Extracting URL: https://www.youtube.com/watch?v=u-ZBLZATlhM
[youtube] u-ZBLZATlhM: Downloading webpage


[youtube] u-ZBLZATlhM: Downloading android vr player API JSON
[info] u-ZBLZATlhM: Downloading 1 format(s): 251
[download] Destination: ../data/audio/erwin_speech_aot.webm
[download] 100% of    1.33MiB in 00:00:00 at 1.91MiB/s     
[ExtractAudio] Destination: ../data/audio/erwin_speech_aot.mp3
Deleting original file ../data/audio/erwin_speech_aot.webm (pass -k to keep)
  Transcribing...
  1297 chars — language: en
  Added 4 chunks
  Saved to disk

bad_bunny_hot_ones
  Removed 26 old chunks
[youtube] Extracting URL: https://www.youtube.com/watch?v=HyqEtb5HE50
[youtube] HyqEtb5HE50: Downloading webpage


[youtube] HyqEtb5HE50: Downloading android vr player API JSON
[info] HyqEtb5HE50: Downloading 1 format(s): 251
[download] Destination: ../data/audio/bad_bunny_hot_ones.webm
[download] 100% of   18.80MiB in 00:00:01 at 10.56MiB/s    
[ExtractAudio] Destination: ../data/audio/bad_bunny_hot_ones.mp3
Deleting original file ../data/audio/bad_bunny_hot_ones.webm (pass -k to keep)
  Transcribing...
  13287 chars — language: en
  Added 31 chunks
  Saved to disk

Done. Restart Streamlit to use updated embeddings.
